# Autoresearch-Tenstorrent Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.
Adapted from upstream `karpathy/autoresearch` for TT-XLA on Tenstorrent N300.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_bpb, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Val BPB Over Time

Track how the best (kept) val_bpb evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

if len(valid) > 0:
    baseline_bpb = valid.loc[0, "val_bpb"]

    # Only plot points at or below baseline (the interesting region)
    below = valid[valid["val_bpb"] <= baseline_bpb + 0.0005]

    # Plot discarded as faint background dots
    disc = below[below["status"] == "DISCARD"]
    ax.scatter(disc.index, disc["val_bpb"],
               c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

    # Plot kept experiments as prominent green dots
    kept_v = below[below["status"] == "KEEP"]
    ax.scatter(kept_v.index, kept_v["val_bpb"],
               c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

    # Running minimum step line
    kept_mask = valid["status"] == "KEEP"
    kept_idx = valid.index[kept_mask]
    kept_bpb = valid.loc[kept_mask, "val_bpb"]
    running_min = kept_bpb.cummin()
    ax.step(kept_idx, running_min, where="post", color="#27ae60",
            linewidth=2, alpha=0.7, zorder=3, label="Running best")

    # Label each kept experiment with its description
    for idx, bpb in zip(kept_idx, kept_bpb):
        desc = str(valid.loc[idx, "description"]).strip()
        if len(desc) > 45:
            desc = desc[:42] + "..."
        ax.annotate(desc, (idx, bpb), textcoords="offset points",
                    xytext=(8, -12), fontsize=7, color="#555555")

    best = running_min.iloc[-1]
    margin = (baseline_bpb - best) * 0.15
    ax.set_ylim(best - margin, baseline_bpb + margin)

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("val_bpb (lower is better)", fontsize=12)
ax.set_title("autoresearch-tenstorrent · N300 TT-XLA optimization progress", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig("progress.png", dpi=150)
plt.show()

if len(valid) > 0:
    print(f"\nBest val_bpb: {running_min.iloc[-1]:.6f}")
    print(f"Baseline:     {baseline_bpb:.6f}")
    print(f"Improvement:  {baseline_bpb - running_min.iloc[-1]:.6f} ({(baseline_bpb - running_min.iloc[-1]) / baseline_bpb:.1%})")

## TT-Specific Notes

- `memory_gb` reports `-1.0` on TT (VRAM monitoring not available via TT-XLA).
- `mfu_percent` is not comparable to upstream H100 values.
- Runs use `AUTORESEARCH_PROFILE=tt_singlechip` on a single Wormhole N300 chip.
- See `docs/KNOWN_LIMITATIONS.md` for features blocked by the TT-XLA stack.